In [ ]:
import importlib
import sys
from pathlib import Path

import pandas as pd
import plotly.io as pio
from IPython.display import display

# Interactive Plotly charts in Jupyter / VS Code / Cursor
pio.renderers.default = "plotly_mimetype+notebook_connected"

sys.path.insert(0, str(Path("..").resolve()))
sys.path.insert(0, str(Path(".").resolve()))

import plotter
importlib.reload(plotter)
from plotter import Scale, plot_series

from engine import EntryType, latest_research_entries, steps_path

OUTCOMES_DIR = Path("../outcomes")
RESEARCH_NAME = "hold-vs-buybelow"


def load_research(research: str = RESEARCH_NAME) -> dict[str, pd.DataFrame]:
    """Load steps for every experiment in the latest batch of `research`."""
    entries = latest_research_entries(OUTCOMES_DIR, research)
    outcomes = {}
    for entry in entries:
        key = entry.get("name") or entry["id"]
        outcomes[key] = pd.read_json(steps_path(OUTCOMES_DIR, entry), lines=True)
    return outcomes

outcomes = load_research()

# for name, steps in outcomes.items():
#     print(name)
#     display(steps)

plot_series(outcomes, "equity", journal=[EntryType.DEPOSIT, EntryType.ORDER_FILLED])


In [ ]:
TICKER = "BTC"
# Same market data across experiments — high/low once from any run
steps = next(iter(outcomes.values()))
df = steps.copy()
df["high"] = df["candles"].apply(lambda c: float(c[TICKER]["high"]))
df["low"] = df["candles"].apply(lambda c: float(c[TICKER]["low"]))
high_df = df.copy()
high_df["level"] = df["high"]
low_df = df.copy()
low_df["level"] = df["low"]
plot_series({"high": high_df, "low": low_df}, "level")

In [ ]:
cash_frames = {}
for name, steps in outcomes.items():
    df = steps.copy()
    df["available_usd"] = df["balances"].apply(lambda b: float(b["USD"]))
    df["frozen_usd"] = df["frozen_usd"].fillna(0).astype(float)
    df["total_usd"] = df["available_usd"] + df["frozen_usd"]
    cash_frames[name] = df

plot_series(cash_frames, "total_usd", scale=Scale.LINEAR)
plot_series(cash_frames, "available_usd", scale=Scale.LINEAR)
plot_series(cash_frames, "frozen_usd", scale=Scale.LINEAR)
